# Assignment 01: ResNet 18 vs ViT

**Available:** Aug 19, 2025 4:30pm until Aug 28, 2025 11:59pm

## Tasks:

1. Train on Human Action Recognition (40%) https://www.kaggle.com/datasets/shashankrapolu/human-Links to an external site.action-recognition-dataset/dataLinks to an external site.​
2. Evaluation on test set human-action-recognition​: 15 classes, 218 MB, 12600 images, train/test split available​
3. ResNet 18 vs ViT​
4. Report and code zip (10%)​
5. Errors and obstacles faced running the model​
6. Must have conda requirement.txt, cli commands to generate the evaluation results above (random checks will be perform to verify)​
7. Training log​
8. Video of live demo (20%)​
9. The video should comprise of visual outputs (e.g., a text "cat" overlay on the image being classified that has a cat in it)​
10. Insights (30%)​
11. ResNet vs ViT​ Computational comparisons​
12. Why one is better ...


### Import Libraries

In [1]:
## Import Libraries

# Set CUDA_VISIBLE_DEVICES to make both GPUs visible
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

import torch
import torch.nn as nn
import torchvision
import cv2
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch import optim
from tqdm.notebook import tqdm
from torchinfo import summary
import einops
import PIL
import numpy as np
import pandas as pd

# Install einops for tensor manipulation
%pip install einops

# Authorize Huggingface account
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv('/mnt/Storage02/SoftwareDev/CAP_6411_Assignments/.env')

# Get Hugging Face token
hf_token = os.getenv('HUGGINGFACE_HUB_TOKEN') or os.getenv('HF_TOKEN')

if hf_token:
    print("🔐 Found Hugging Face token in environment variables")
    
    # Install huggingface_hub if not already installed
    %pip install huggingface_hub
    
    from huggingface_hub import login, whoami
    
    try:
        # Login to Hugging Face Hub
        login(token=hf_token)
        
        # Verify login by getting user info
        user_info = whoami()
        print(f"✅ Successfully authenticated with Hugging Face!")
        print(f"👤 Logged in as: {user_info['name']}")
        
        # Set the token as environment variable for other libraries
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
        os.environ['HF_TOKEN'] = hf_token
        
    except Exception as e:
        print(f"❌ Authentication failed: {e}")
        print("⚠️  Will proceed without pre-trained models if needed")
        hf_token = None
else:
    print("⚠️  No Hugging Face token found in .env file")
    print("📝 Please add HUGGINGFACE_HUB_TOKEN=your_token_here to your .env file")
    hf_token = None

# Comprehensive GPU diagnostics
print("\n=== GPU Diagnostics ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print("\n=== All Available GPUs ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Multi-processor count: {props.multi_processor_count}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print()

# Device selection with preference for cuda:1 (A6000) -> cuda:0 (4090) -> cpu
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = torch.device('cuda:1')  # This should now be your A6000!
    print(f"Using GPU 1: {torch.cuda.get_device_name(1)}")
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Selected device: {device}")

Note: you may need to restart the kernel to use updated packages.
🔐 Found Hugging Face token in environment variables
Note: you may need to restart the kernel to use updated packages.
🔐 Found Hugging Face token in environment variables
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Successfully authenticated with Hugging Face!
👤 Logged in as: malneyugnfl

=== GPU Diagnostics ===
PyTorch version: 2.5.1
CUDA available: True
CUDA version: 12.4
Number of GPUs detected: 2

=== All Available GPUs ===
GPU 0:
  Name: NVIDIA GeForce RTX 4090 Laptop GPU
  Total Memory: 15.70 GB
  Multi-processor count: 76
  Compute Capability: 8.9

GPU 1:
  Name: NVIDIA RTX A6000
  Total Memory: 47.53 GB
  Multi-processor count: 84
  Compute Capability: 8.6

Using GPU 1: NVIDIA RTX A6000
Selected device: cuda:1


### Import and Process Human Action Recognition Dataset

In [2]:
## Import dataset and process it

class ActionRecognitionDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        """
        Custom dataset for Human Action Recognition with labels
        Args:
            csv_file (string): Path to the csv file with annotations.
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        import csv
        
        self.root_dir = root_dir
        self.transform = transform
        
        # Read CSV manually to avoid pandas compatibility issues
        self.annotations = []
        with open(csv_file, 'r') as file:
            reader = csv.reader(file)
            headers = next(reader)  # Skip header
            for row in reader:
                self.annotations.append({
                    'filename': row[0],
                    'label': row[1]
                })
        
        # Get unique class names and create class-to-index mapping
        unique_labels = set(row['label'] for row in self.annotations)
        self.classes = sorted(list(unique_labels))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        
        print(f"Found {len(self.classes)} classes: {self.classes}")
        print(f"Total samples: {len(self.annotations)}")
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
            
        # Get image filename and label from annotations
        annotation = self.annotations[idx]
        img_name = annotation['filename']
        label_name = annotation['label']
        
        # Convert label name to index
        label = self.class_to_idx[label_name]
        
        # Construct full image path
        img_path = os.path.join(self.root_dir, img_name)
        
        # Load image
        image = PIL.Image.open(img_path)
        
        # Convert RGBA to RGB if necessary
        if image.mode == 'RGBA':
            image = image.convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, label

class TestDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        """
        Test dataset for Human Action Recognition without labels (for prediction)
        Args:
            csv_file (string): Path to the csv file with filenames only.
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        import csv
        
        self.root_dir = root_dir
        self.transform = transform
        
        # Read CSV manually to avoid pandas compatibility issues
        self.annotations = []
        with open(csv_file, 'r') as file:
            reader = csv.reader(file)
            headers = next(reader)  # Skip header
            for row in reader:
                self.annotations.append({
                    'filename': row[0]
                })
        
        print(f"Found {len(self.annotations)} test images")
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
            
        # Get image filename from annotations
        annotation = self.annotations[idx]
        img_name = annotation['filename']
        
        # Construct full image path
        img_path = os.path.join(self.root_dir, img_name)
        
        # Load image
        image = PIL.Image.open(img_path)
        
        # Convert RGBA to RGB if necessary
        if image.mode == 'RGBA':
            image = image.convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, img_name  # Return image and filename for prediction

# Data transforms with proper normalization for RGB images
data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),  # Converts PIL to tensor and scales to [0,1]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

# Define data augmentation for training
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Dataset classes updated to avoid pandas compatibility issues!")

✅ Dataset classes updated to avoid pandas compatibility issues!


In [3]:
# Fix for pandas CSV reading issue
# This error often occurs due to version incompatibility between pandas and numpy
# Let's try reading the CSV with a different engine or method

import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

# Try to read a sample CSV to test if the issue persists
try:
    test_df = pd.read_csv('data/Training_set.csv', engine='python')
    print(f"✅ CSV reading test successful! Found {len(test_df)} rows")
    print(f"Columns: {list(test_df.columns)}")
    print(f"First 3 rows:\n{test_df.head(3)}")
except Exception as e:
    print(f"❌ CSV reading still fails: {e}")
    print("Trying alternative solution...")

Pandas version: 2.3.2
NumPy version: 2.2.6
✅ CSV reading test successful! Found 12600 rows
Columns: ['filename', 'label']
First 3 rows:
      filename         label
0  Image_1.jpg       sitting
1  Image_2.jpg  using_laptop
2  Image_3.jpg       hugging


In [4]:
# Create datasets
train_data = ActionRecognitionDataset(
    csv_file='data/Training_set.csv',
    root_dir='data/train',
    transform=train_transform
)

# For this assignment, we'll create validation data from training data
# Split training data into train and validation (80-20 split)
from torch.utils.data import random_split

train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size
train_subset, val_subset = random_split(train_data, [train_size, val_size])

# Create validation dataset with test transforms (no augmentation)
val_data = ActionRecognitionDataset(
    csv_file='data/Training_set.csv',
    root_dir='data/train',
    transform=data_transform
)

# Extract validation indices for proper subset
val_dataset = torch.utils.data.Subset(val_data, val_subset.indices)

# Create test dataset and loader for final evaluation
test_data = TestDataset(
    csv_file='data/Testing_set.csv',
    root_dir='data/test',
    transform=data_transform  # No augmentation for test data
)

print(f"\nDataset Information:")
print(f"Total training samples: {len(train_data)}")
print(f"Training subset: {len(train_subset)}")
print(f"Validation subset: {len(val_dataset)}")
print(f"Number of classes: {len(train_data.classes)}")
print(f"Classes: {train_data.classes}")
print(f"\nTest Dataset Information:")
print(f"Test samples: {len(test_data)}")


# Test loading a sample
try:
    sample_image, sample_label = train_data[0]
    print(f"\nSample check:")
    print(f"Image shape: {sample_image.shape}")
    print(f"Label: {sample_label} ({train_data.classes[sample_label]})")
except Exception as e:
    print(f"Error loading sample: {e}")
    print("Please check if the image files exist in the correct directory")

# Test loading from test dataset
try:
    test_sample_image, test_filename = test_data[0]
    print(f"\nTest Sample Check:")
    print(f"Test image shape: {test_sample_image.shape}")
    print(f"Test filename: {test_filename}")
    print("Test dataset loading successful!")
except Exception as e:
    print(f"Error loading test sample: {e}")
    print("Please check if the test image files exist in the correct directory")

Found 15 classes: ['calling', 'clapping', 'cycling', 'dancing', 'drinking', 'eating', 'fighting', 'hugging', 'laughing', 'listening_to_music', 'running', 'sitting', 'sleeping', 'texting', 'using_laptop']
Total samples: 12600
Found 15 classes: ['calling', 'clapping', 'cycling', 'dancing', 'drinking', 'eating', 'fighting', 'hugging', 'laughing', 'listening_to_music', 'running', 'sitting', 'sleeping', 'texting', 'using_laptop']
Total samples: 12600
Found 5400 test images

Dataset Information:
Total training samples: 12600
Training subset: 10080
Validation subset: 2520
Number of classes: 15
Classes: ['calling', 'clapping', 'cycling', 'dancing', 'drinking', 'eating', 'fighting', 'hugging', 'laughing', 'listening_to_music', 'running', 'sitting', 'sleeping', 'texting', 'using_laptop']

Test Dataset Information:
Test samples: 5400

Sample check:
Image shape: torch.Size([3, 224, 224])
Label: 11 (sitting)

Test Sample Check:
Test image shape: torch.Size([3, 224, 224])
Test filename: Image_1.jpg


### Create Dataloaders that will be used for ResNet 18 and ViT

In [5]:
# Create data loaders
batch_size = 50
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=4)


print(f"\nData Loaders:")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")


Data Loaders:
Training batches: 202
Validation batches: 51
Test batches: 108


### Import ResNet 18

In [6]:
resnet18= torchvision.models.resnet18(weights='IMAGENET1K_V1')
num_ftrs = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_ftrs, len(train_data.classes))  # Adjust final layer for 15 classes
resnet18 = resnet18.to(device)

### Import ViT

In [7]:
# Import and Setup ViT

# Import and Setup ViT
# Install timm (PyTorch Image Models) for Vision Transformer
%pip install timm

import timm
import torch
import torch.nn as nn

# Define number of classes for Human Action Recognition
num_classes = len(train_data.classes)  # Should be 15 classes

print("=== Setting up Vision Transformer ===")

# Try to load a pre-trained Vision Transformer, fallback to non-pretrained if authorization fails
try:
    print("Attempting to load pre-trained ViT model...")
    vit_model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes)
    print("✅ Pre-trained ViT model loaded successfully!")
except Exception as e:
    print(f"⚠️  Could not load pre-trained model: {e}")
    print("📌 Loading ViT without pre-trained weights...")
    vit_model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=num_classes)
    print("✅ ViT model (without pre-training) loaded successfully!")

# Move model to device 
vit_model = vit_model.to(device)

print(f"Model: {vit_model.__class__.__name__}")
print(f"Number of classes: {num_classes}")
print(f"Device: {device}")

# Get model information
vit_params = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
print(f'ViT trainable parameters in millions: {vit_params/1000000:.2f}')

# Create a sample input to test the model
print(f"\n=== Testing ViT with Sample Data ===")
try:
    # Create a dummy batch (batch_size=2, channels=3, height=224, width=224)
    sample_input = torch.randn(2, 3, 224, 224).to(device)
    print(f"Sample input shape: {sample_input.shape}")
    
    # Test forward pass
    with torch.no_grad():
        vit_output = vit_model(sample_input)
    
    print(f"ViT output shape: {vit_output.shape}")
    print(f"Expected shape: (batch_size=2, num_classes={num_classes})")
    print("✅ ViT is ready for training!")
    
    # Display model architecture summary
    print(f"\n=== ViT Architecture Summary ===")
    print(f"Model type: Vision Transformer Base (patch size 16)")
    print(f"Input size: 224x224 pixels") 
    print(f"Patch size: 16x16 pixels")
    print(f"Number of patches: {(224//16)**2} patches")
    print(f"Embedding dimension: 768")
    print(f"Number of attention heads: 12")
    print(f"Number of transformer layers: 12")
    print(f"Output classes: {num_classes}")
    
except Exception as e:
    print(f"❌ Error testing ViT: {e}")

print(f"\n🎯 ViT is now ready to be compared with ResNet-18!")
print(f"Both models will classify images into {num_classes} human action categories:")
print(f"Classes: {train_data.classes}")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
=== Setting up Vision Transformer ===
Attempting to load pre-trained ViT model...
=== Setting up Vision Transformer ===
Attempting to load pre-trained ViT model...
✅ Pre-trained ViT model loaded successfully!
Model: VisionTransformer
Number of classes: 15
Device: cuda:1
ViT trainable parameters in millions: 85.81

=== Testing ViT with Sample Data ===
Sample input shape: torch.Size([2, 3, 224, 224])
✅ Pre-trained ViT model loaded successfully!
Model: VisionTransformer
Number of classes: 15
Device: cuda:1
ViT trainable parameters in millions: 85.81

=== Testing ViT with Sample Data ===
Sample input shape: torch.Size([2, 3, 224, 224])
ViT output shape: torch.Size([2, 15])
Expected shape: (batch_size=2, num_classes=15)
✅ ViT is ready for training!

=== ViT Architecture Summary ===
Model type: Vision Transformer Base (patch size 16)
Input size: 224x224 pixels
P

### Summary of Models

In [8]:
# Test both models with synthetic data (to avoid numpy compatibility issues)
print("🧪 Testing both models with synthetic data...")

# Create synthetic test data
batch_size_test = 2
test_images = torch.randn(batch_size_test, 3, 224, 224).to(device)
test_labels = torch.randint(0, num_classes, (batch_size_test,)).to(device)

print(f"Test batch shape: {test_images.shape}")
print(f"Test labels shape: {test_labels.shape}")

# Test ResNet-18
print("\n🔍 Testing ResNet-18...")
with torch.no_grad():
    resnet_output = resnet18(test_images)
    resnet_probs = torch.softmax(resnet_output, dim=1)
    resnet_preds = torch.argmax(resnet_output, dim=1)
    print(f"ResNet-18 output shape: {resnet_output.shape}")
    print(f"ResNet-18 predictions: {resnet_preds}")
    print(f"ResNet-18 max probabilities: {torch.max(resnet_probs, dim=1)[0]}")

# Test ViT
print("\n🔍 Testing ViT...")
with torch.no_grad():
    vit_output = vit_model(test_images)
    vit_probs = torch.softmax(vit_output, dim=1)
    vit_preds = torch.argmax(vit_output, dim=1)
    print(f"ViT output shape: {vit_output.shape}")
    print(f"ViT predictions: {vit_preds}")
    print(f"ViT max probabilities: {torch.max(vit_probs, dim=1)[0]}")

print("\n✅ Both models are working correctly!")
print(f"📊 Ready to train on {len(train_data.classes)} classes: {train_data.classes}")

# Model comparison summary
print(f"\n📈 Model Comparison Summary:")
resnet_params = sum(p.numel() for p in resnet18.parameters() if p.requires_grad)
vit_params = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
print(f"ResNet-18 parameters: {resnet_params/1e6:.2f}M")
print(f"ViT parameters: {vit_params/1e6:.2f}M")
print(f"Parameter ratio (ViT/ResNet): {vit_params/resnet_params:.1f}x")
print(f"Training samples: {len(train_subset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Test samples: {len(test_data):,}")

# Memory usage info
print(f"\n💾 GPU Memory Usage:")
if torch.cuda.is_available():
    print(f"Allocated: {torch.cuda.memory_allocated(device)/1e9:.2f} GB")
    print(f"Cached: {torch.cuda.memory_reserved(device)/1e9:.2f} GB")

🧪 Testing both models with synthetic data...
Test batch shape: torch.Size([2, 3, 224, 224])
Test labels shape: torch.Size([2])

🔍 Testing ResNet-18...
ResNet-18 output shape: torch.Size([2, 15])
ResNet-18 predictions: tensor([2, 2], device='cuda:1')
ResNet-18 max probabilities: tensor([0.2409, 0.2354], device='cuda:1')

🔍 Testing ViT...
ViT output shape: torch.Size([2, 15])
ViT predictions: tensor([3, 3], device='cuda:1')
ViT max probabilities: tensor([0.1042, 0.1157], device='cuda:1')

✅ Both models are working correctly!
📊 Ready to train on 15 classes: ['calling', 'clapping', 'cycling', 'dancing', 'drinking', 'eating', 'fighting', 'hugging', 'laughing', 'listening_to_music', 'running', 'sitting', 'sleeping', 'texting', 'using_laptop']

📈 Model Comparison Summary:
ResNet-18 parameters: 11.18M
ViT parameters: 85.81M
Parameter ratio (ViT/ResNet): 7.7x
Training samples: 10,080
Validation samples: 2,520
Test samples: 5,400

💾 GPU Memory Usage:
Allocated: 0.40 GB
Cached: 0.47 GB


### Run Training for ViT and ResNet-18

In [9]:
# Run Training for Vit and ResNet-18

In [10]:
# Compare results and do analysis